In [1]:
import os
import re
import pickle
from tqdm import tqdm
from unicodedata import normalize

import multiprocessing
import concurrent.futures
from functools import lru_cache

import numpy as np
import pandas as pd

import requests
from bs4 import BeautifulSoup

from jellyfish import jaro_winkler_similarity

import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

print(f"Num workers: {multiprocessing.Pool()._processes}")

Num workers: 12


In [2]:
def punct(full_text):
    temp = [sent.strip() for sent in re.findall("""\s+[^.!?]*[.!?]""", full_text)]
    temp = re.sub('\..', '.', '. '.join(temp))
    temp = re.sub('\s+[a-zA-Z]\.', '', temp)
    
    return temp

In [3]:
start_page = 0
end_page = 4000
increment = 200

search_term = 'audio+recognition'
base_link = f"https://arxiv.org/search/advanced?advanced=&terms-0-operator=AND&terms-0-term={search_term}&terms-0-field=all&classification-physics_archives=all&classification-include_cross_list=include&date-filter_by=all_dates&date-year=&date-from_date=&date-to_date=&date-date_type=submitted_date&abstracts=show&size=200&order=-announced_date_first"

papers = []

for start_num in tqdm(range(start_page, end_page, increment), desc = f'Scraping papers from arxiv using {search_term} search term'):
    linkie = f"{base_link}&start={start_num}"
    
    html_text = requests.get(linkie).text
    soup = BeautifulSoup(html_text)
    
    papers_sub = soup.find_all('li', class_ = 'arxiv-result')
    papers += papers_sub
    
print(f"Number of papers scraped: {len(papers)}")

Scraping papers from arxiv using audio+recognition search term: 100%|██████████| 20/20 [00:53<00:00,  2.68s/it]

Number of papers scraped: 4000


In [4]:
arxiv_code = []
titles = []
authors = []
abstracts = []
journal = []
submitted_dates = []
originally_announced_dates = []

for paper in tqdm(papers, desc = 'Scraping fields'):
    paper_arxiv_code = paper.find('p', class_ = 'list-title is-inline-block').text.split('\n')[0]

    paper_title = paper.find('p', class_ = 'title is-5 mathjax').text.strip()
    
    paper_authors = paper.find('p', class_ = 'authors')
    paper_authors = ','.join([name.text.strip() for name in paper_authors.find_all('a')])

    paper_abstract = paper.find('p', class_ = 'abstract mathjax')
    paper_abstract = punct(paper_abstract.find('span', class_ = 'abstract-full has-text-grey-dark mathjax').text)

    paper_comment = paper.find('p', class_='comments is-size-7')
    if paper_comment:
        paper_comment_text = re.sub(r'\s+', ' ', paper_comment.text.strip())
        patterns = {
            'submitted to': r'submitted to (.+)',
            'Accepted to': r'Accepted to (.+)',
            'Accepted in': r'Accepted in (.+)',
            'accepted by': r'accepted by (.+)',
            'Journal ref': r'Journal ref: (.+)'
        }
        for keyword, pattern in patterns.items():
            if keyword in paper_comment_text:
                match = re.search(pattern, paper_comment_text)
                if match:
                    paper_journal = match.group(1).strip()
                    break
    else:
        paper_journal = np.nan
    
    submit_date = paper.find('p', class_ = 'is-size-7').text.split(';')[0]
    submit_date = re.findall('([0-9]{1,2}\s(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|(Nov|Dec)(?:ember)?)\, [0-9]{4})', submit_date)
    submit_date = [sent for sent in submit_date[0] if len(sent)][0]

    originally_announced_date = paper.find('p', class_ = 'is-size-7').text.strip().split('originally announced ')[-1][:-1]
    
    arxiv_code.append(paper_arxiv_code)
    titles.append(paper_title)
    authors.append(paper_authors)
    abstracts.append(paper_abstract)
    journal.append(paper_journal)
    submitted_dates.append(submit_date)
    originally_announced_dates.append(originally_announced_date)

Scraping fields: 100%|██████████| 4000/4000 [00:03<00:00, 1287.01it/s]


In [5]:
df = pd.DataFrame({'title': titles, 
                   'authors': authors,
                   'abstract': abstracts,
                   'Journal': journal,
                   'code': arxiv_code,
                   'submitted_date': submitted_dates,
                   'orginally_announced_date': originally_announced_dates})
df['submitted_date'] = pd.to_datetime(df['submitted_date'])
df['orginally_announced_date'] = pd.to_datetime(df['orginally_announced_date'], errors = 'coerce')

df.head()

,title,authors,abstract,Journal,code,submitted_date,orginally_announced_date
0,Speech Emotion Recognition Via CNN-Transforemr...,"Xiaoyu Tang,Yixin Lin,Ting Dang,Yuanfang Zhang...",Speech Emotion Recognition (SER) is crucial in...,NaN,arXiv:2403.04743,2024-03-07,2024-03-01
1,Dynamic Cross Attention for Audio-Visual Perso...,"R. Gnana Praveen,Jahangir Alam",Although person or identity verification has b...,FG2024,arXiv:2403.04661,2024-03-07,2024-03-01
2,Audio-Visual Person Verification based on Recu...,"R. Gnana Praveen,Jahangir Alam",Person or identity verification has been recen...,FG2024,arXiv:2403.04654,2024-03-07,2024-03-01
3,CAT: Enhancing Multimodal Large Language Model...,"Qilang Ye,Zitong Yu,Rui Shao,Xinyu Xie,Philip ...",This paper focuses on the challenge of answeri...,NaN,arXiv:2403.04640,2024-03-07,2024-03-01
4,A New Benchmark for Evaluating Automatic Speec...,"Qusai Abo Obaidah,Muhy Eddin Zater,Adnan Jalju...",This work is an attempt to introduce a compreh...,NaN,arXiv:2403.04280,2024-03-07,2024-03-01


In [6]:
df.info(memory_usage = 'deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   title                     4000 non-null   object        
 1   authors                   4000 non-null   object        
 2   abstract                  4000 non-null   object        
 3   Journal                   1547 non-null   object        
 4   code                      4000 non-null   object        
 5   submitted_date            4000 non-null   datetime64[ns]
 6   orginally_announced_date  4000 non-null   datetime64[ns]
dtypes: datetime64[ns](2), object(5)
memory usage: 6.3 MB


In [7]:
output_filename = f"arxiv_{search_term}_search_term.parquet"
df.to_parquet(output_filename)

# Scraping authors info

In [8]:
author_pairs = []

for authors in df['authors']:
    author_list = authors.split(',')
    author_pairs.extend([(author.strip(), other_author.strip()) for i, author in enumerate(author_list) for other_author in author_list[i+1:]])

author_pair_df = pd.DataFrame(author_pairs, columns = ['Author1', 'Author2'])

print(f"Authors network shape: {author_pair_df.shape}")
author_pair_df.head()

Authors network shape: (59222, 2)


,Author1,Author2
0,Xiaoyu Tang,Yixin Lin
1,Xiaoyu Tang,Ting Dang
2,Xiaoyu Tang,Yuanfang Zhang
3,Xiaoyu Tang,Jintao Cheng
4,Yixin Lin,Ting Dang


In [9]:
session = requests.Session()

@lru_cache(maxsize = None)
def return_matching_profile_link(base_link, author, cut_off=0.9):
    author_search_name = author.replace(' ', '+')
    linkie = f'https://scholar.google.com/citations?hl=en&view_op=search_authors&mauthors={author_search_name}&btnG='

    html_text = session.get(linkie).text
    soup = BeautifulSoup(html_text, 'html.parser')

    profile = soup.find_all('div', class_='gsc_1usr')

    jaro_similarity_scores = []
    hit_names = []
    links = []

    if profile:
        for hit in profile:
            hit = hit.find('h3', class_='gs_ai_name').find('a')
            hit_name = hit.text.strip()
            hit_link = base_link + hit['href']

            jaro_similarity_score = jaro_winkler_similarity(author, hit_name)

            if jaro_similarity_score >= cut_off:
                jaro_similarity_scores.append(jaro_similarity_score)
                links.append(hit_link)
                hit_names.append(hit_name)

    if len(jaro_similarity_scores):
        max_score_ind = np.argmax(jaro_similarity_scores)
        max_score_profile_name = hit_names[max_score_ind]
        profile_link = links[max_score_ind]
    else:
        profile_link = None
        max_score_profile_name = np.nan

    return profile_link, max_score_profile_name

In [10]:
@lru_cache(maxsize = None)
def return_author_info(profile_link, name):
    if profile_link:
        user_html_text = session.get(profile_link).text
        user_soup = BeautifulSoup(user_html_text, 'html.parser')

        institute = user_soup.find('div', class_='gsc_prf_il')
        research_area = user_soup.find('div', {"id": "gsc_prf_int"}, class_='gsc_prf_il')

        if institute:
            institute = institute.text.strip()
        if research_area:
            research_area = [item.text.lower().strip() for item in research_area.find_all('a')]
        elif (not institute) and (not research_area):
            institute = np.nan
            research_area = np.nan
    else:
        institute = np.nan
        research_area = np.nan

    return institute, research_area

In [11]:
hit_names = []
author_institute = []
author_research_area = []
author_unique_names = pd.concat([author_pair_df['Author1'], author_pair_df['Author2']]).unique().tolist()

base_link = 'https://scholar.google.com'

def process_author(name):
    profile_link, min_score_profile_name = return_matching_profile_link(base_link, name)
    institute, research_area = return_author_info(profile_link, name)

    if isinstance(research_area, list):
        research_area = ','.join(research_area)

    hit_names.append(min_score_profile_name)
    author_institute.append(institute)
    author_research_area.append(research_area)

In [12]:
num_workers = 8
with concurrent.futures.ThreadPoolExecutor(max_workers = num_workers) as executor:
    future_to_author = {executor.submit(process_author, name): name for name in author_unique_names}

    for future in tqdm(concurrent.futures.as_completed(future_to_author), total = len(future_to_author), desc = "Processing Authors"):
        author_name = future_to_author[future]
        try:
            future.result()
        except Exception as e:
            print(f"Error processing author {author_name}: {e}")

Processing Authors: 100%|██████████| 10618/10618 [11:05<00:00, 15.94it/s]


In [13]:
with open('author_institute.pickle', 'wb') as handle:
    pickle.dump(author_institute, handle, protocol = pickle.HIGHEST_PROTOCOL)
    print("saved to author_institute.pickle")

saved to author_institute.pickle


In [14]:
with open('author_research_area.pickle', 'wb') as handle:
    pickle.dump(author_research_area, handle, protocol = pickle.HIGHEST_PROTOCOL)
    print("saved to author_research_area.pickle")

saved to author_research_area.pickle


In [15]:
authors_info = pd.DataFrame({'author': author_unique_names,
                             'hit_name': hit_names,
                             'institute': author_institute,
                             'research_area': author_research_area})

output_filename = f"arxiv_{search_term}_search_term_authors_info.parquet"
authors_info.to_parquet(output_filename)